# 01 — Explore the evidence

**No training happens here.** This notebook reads the eleven committed runs and
asks where retrieval actually fails, so that whatever you build next is aimed at
a real failure mode rather than an assumed one.

That ordering matters in this repo specifically. Three tickets were written
against premises that later measurement falsified:

- **RD-14** existed because dictionary-phrased queries beat human-phrased ones by
  ~43 points. On the index that actually serves users that gap is **+1.8pp** —
  RD-02 closed it for free, by changing what was indexed rather than what the
  model was trained on.
- **RD-09** assumed more/better training data was the lever. Its own proxy
  experiment (RD-16) lost **7.0 points**.
- **RD-12/RD-13** assumed a reranker was ~53 points away. RD-21 measured the
  gaps and found they are mostly *not* near-ties.

Every one of those was cheap to check and expensive to assume.

In [ ]:
import sys, os
from pathlib import Path

# rdlib lives at training/rdlib; this notebook is at training/notebooks.
sys.path.insert(0, str(Path.cwd().parent))

# Cells are large and the darwin default lives under os.tmpdir(), which gets
# reaped. Point this somewhere durable and OUTSIDE the repo -- the working tree
# is in OneDrive, which would try to sync ~170 MB per cell.
os.environ.setdefault("EVAL_CELL_DIR", str(Path.home() / "rd_eval_cells"))

import rdlib
from rdlib import paths
print("repo      ", paths.REPO_ROOT)
print("cells     ", paths.cell_dir())

In [ ]:
import pandas as pd
from rdlib import runs as R
from rdlib.metrics import score, compare

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

available = R.list_runs()
print(f"{len(available)} committed runs:\n")
for t in available:
    print(" ", t)

## The headline table, recomputed

Not transcribed from CLAUDE.md — recomputed from each run's own rows, on the
287-row authored-reachable slice. If a figure here disagrees with the docs, the
docs are what is wrong.

In [ ]:
def headline_rows(run):
    return [r for r in run.results if r.source == "authored" and r.meta.get("reachable")]

rows = []
for tag in available:
    run = R.load_run(tag)
    hr = headline_rows(run)
    if not hr:
        continue
    m = score(hr)
    rows.append({
        "run": tag,
        "n": m.n,
        "lenient R@1": round(m.lenient_recall1 * 100, 1),
        "strict R@1": round(m.recall1 * 100, 1),
        "R@10": round(m.recall10 * 100, 1),
        "MRR@10": round(m.mrr10, 3),
        "echo %": round(m.echo_rate * 100, 1),
        "index": run.config.get("index", "?"),
        "probes": run.config.get("probes"),
    })

pd.DataFrame(rows).sort_values("lenient R@1", ascending=False).reset_index(drop=True)

## Where the failures are

This is the single most-cited number in the backlog, and RD-21 exists because it
was being read wrongly. Split the production run by where the target ended up.

In [ ]:
prod = R.load_run("prod_wikt_shipped")
hr = headline_rows(prod)
n = len(hr)

def band(r):
    if r.lenient_rank is None:      return "never retrieved (within 100)"
    if r.lenient_rank == 1:         return "rank 1  — already correct"
    if r.lenient_rank <= 10:        return "ranks 2-10  — reordering territory"
    return "ranks 11-100  — deep in the shortlist"

counts = pd.Series([band(r) for r in hr]).value_counts()
summary = pd.DataFrame({"queries": counts, "share %": (counts / n * 100).round(1)})
print(f"n = {n} (authored, reachable)  --  run: {prod.tag}\n")
summary

**These are not the numbers the backlog quotes, and that is expected.** The
figures repeated across RD-09/12/13/21 — *"24.0% at rank 1, 77.0% within 100"* —
come from `prod_gloss_shipped`, the **pre-RD-17** index. RD-17's vocabulary
expansion moved them: within-100 went 77.0% → **83.6%**, and never-retrieved
fell 23.0% → **16.4%**.

If you cite a headroom number, cite it from the run you actually measured,
and say which run that was.

Read that carefully. The middle two bands are what every reranking ticket calls
"headroom": the answer *was* retrieved, just not put first.

**RD-21's correction, and it is the thing to internalise:** being in the
shortlist is not the same as being a near-miss. Of the targets that are in the
shortlist but not first, near-ties (similarity gap < 0.03) are only **26.2%** —
**73.8% lose by a confident margin**, median gap 0.061. The gloss cutover
roughly *halved* the lemma index's margin (+0.094 → +0.070) and did not close
it.

So a reranker's job here was never "break ties". That is a second, independent
reason RD-12's cross-encoder failed.

Below is the part of that computable from a committed run — targets at ranks
2–10, where the run stored the similarities.

In [ ]:
gaps = []
for r in hr:
    if r.lenient_rank and 2 <= r.lenient_rank <= 10 and len(r.similarities) >= r.lenient_rank:
        gaps.append({
            "id": r.id,
            "target": r.target,
            "rank": r.lenient_rank,
            "winner": r.results[0],
            "winner_sim": r.similarities[0],
            "target_sim": r.similarities[r.lenient_rank - 1],
            "gap": r.similarities[0] - r.similarities[r.lenient_rank - 1],
        })

g = pd.DataFrame(gaps)
print(f"{len(g)} queries with the target at ranks 2-10\n")
print(f"  median gap      {g['gap'].median():.3f}")
print(f"  mean winner sim {g['winner_sim'].mean():.3f}")
print(f"  mean target sim {g['target_sim'].mean():.3f}")
print(f"  near-ties (<0.03) {(g['gap'] < 0.03).mean() * 100:.1f}%")
print()
print("CAVEAT -- do not read that near-tie share as RD-21's 26.2%.")
print("Restricting to ranks 2-10 SELECTS THE CLOSEST MISSES, so the near-tie")
print("share here is biased upward against the full depth-100 population.")
print("RD-21 measures all of it: `npm run probe:margin` at the repo root.")
g.nlargest(8, "gap")[["target", "winner", "rank", "gap"]]

### The caveat that must travel with that number

The widest gap in the set is `merriness` losing to `cheerfulness` — where the
model is arguably **right** and the answer key is incomplete. Only 133 of 312
authored rows carry an `acceptable[]` list, so on the other 179 lenient R@1
collapses to strict.

**73.8% is therefore an upper bound on the model's fault, not a measurement of
it.** Repairing the answer key is the cheapest way to sharpen every future
experiment — and it means a new `v2.jsonl`, because the set is frozen.

In [ ]:
from rdlib.evalset import load_eval_set, authored

ev = load_eval_set()
a = authored(ev)
with_acc = [r for r in a if r.acceptable]
print(f"authored rows              {len(a)}")
print(f"  with acceptable[]        {len(with_acc)}  ({len(with_acc)/len(a)*100:.0f}%)")
print(f"  lenient == strict on     {len(a) - len(with_acc)} rows")

## Slices

`meta` carries the axes the set was built to be sliced on. Frequency **bands**
are derived here rather than stored, so their boundaries can be redrawn without
rebuilding the set.

In [ ]:
df = R.to_dataframe(prod)
df = df[(df["source"] == "authored") & (df["reachable"] == True)]

def slice_table(col):
    g = df.groupby(col).agg(
        n=("id", "size"),
        lenient_R1=("lenient_hit1", lambda s: round(s.mean() * 100, 1)),
        R10=("hit10", lambda s: round(s.mean() * 100, 1)),
        echo=("echo", lambda s: round(s.mean() * 100, 1)),
    )
    return g.sort_values("n", ascending=False)

for col in ["style", "token_count", "lexical_overlap"]:
    print(f"\n--- by {col} ---")
    print(slice_table(col))

In [ ]:
import numpy as np

# Zipf is stored raw (OpenSubtitles 2018). Bands are an analysis-time choice.
# That corpus is conversational, so it under-weights literary and technical
# vocabulary: "rare" here is not rare in writing.
df["freq_band"] = pd.cut(
    df["zipf"], bins=[0, 2.5, 3.5, 4.5, 5.5, 10],
    labels=["very rare", "rare", "mid", "common", "very common"],
)
print(slice_table("freq_band"))

## Echo

Echo is a **primary** metric, not a diagnostic. The standing rule in CLAUDE.md
is that a change improving recall without moving echo needs explaining — and
RD-12 rejected an arm that bought recall by driving echo from 14.5% to 21.4%.

In [ ]:
from rdlib.echo import content_tokens, echoes_query

worst = sorted(hr, key=lambda r: -r.echo)[:6]
for r in worst:
    toks = content_tokens(r.query)
    marked = [f"{w}*" if echoes_query(w, toks) else w for w in r.results[:6]]
    print(f"echo {r.echo:.0%}  target={r.target!r}")
    print(f"   q: {r.query}")
    print(f"   -> {', '.join(marked)}\n")

## The pre-cutover comparison, re-derived

`baseline` is the lemma index — what the app served before RD-02. This is the
largest measured win in the project, and it came from changing **what was
indexed**, with no retraining at all.

In [ ]:
for before, after in [("baseline", "prod_gloss_shipped"),
                      ("prod_gloss_shipped", "prod_wikt_shipped")]:
    a_, b_ = R.load_run(before), R.load_run(after)
    c = compare(headline_rows(a_), headline_rows(b_), lenient=True)
    verdict = "CLEARS the 9a bar" if c["clears_9a_bar"] else "null result under 9a"
    print(f"{before:22} -> {after:20} {c['delta_pp']:+6.1f}pp  "
          f"{c['n_wins']:>3}W/{c['n_regressions']:<3}R  p={c['p']:.4f}   {verdict}")

## What this says about where to aim

1. **Changing indexed text is the best-measured lever in this repo** (+12.9pp),
   and it needs no GPU. `rdlib.wordnet.gloss_text_for()` is one function with
   three branches; a fourth is a complete experiment. Start here if you want a
   result today.
2. **The shortlist headroom is real but is not near-misses.** A reranker has to
   genuinely rescore, not tie-break.
3. **The answer key is half-built**, and it caps how sharply anything can be
   measured.
4. **Retraining has the weakest prior** — the last attempt bought +4.5pp against
   its own base, and the corpus bet lost 7.0pp by proxy.

**Next:** `02_biencoder_finetune.ipynb` or `03_crossencoder_finetune.ipynb`.
Read the header of whichever you pick — both open with the measurements that
constrain them.